In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve
from sklearn.calibration import calibration_curve

# Predictions
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label='ROC curve')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# Calibration Plot
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10)
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('Predicted Probability')
plt.ylabel('True Probability')
plt.title('Calibration Plot')
plt.show()

## 5. Model Evaluation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42, stratify=y)

# Models
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss')
}

best_model = None
best_score = 0

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    mean_score = np.mean(scores)
    print(f"{name} CV AUC: {mean_score:.4f}")

    if mean_score > best_score:
        best_score = mean_score
        best_model = model

# Train best model
best_model.fit(X_train, y_train)
print(f"Best model: {best_model.__class__.__name__}")

## 4. Model Training & Validation

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# LASSO for feature selection
lasso = LassoCV(cv=5, random_state=42)
lasso.fit(X_scaled, y)

print("Best alpha:", lasso.alpha_)

selector = SelectFromModel(lasso, prefit=True)
X_selected = selector.transform(X_scaled)
selected_features = X.columns[selector.get_support()]

print("Selected features:", selected_features.tolist())

In [ ]:
# Feature Engineering
# Create additional features
X_engineered = X.copy()

# Compactness: perimeter^2 / area
X_engineered['mean compactness'] = X_engineered['mean perimeter']**2 / X_engineered['mean area']
X_engineered['worst compactness'] = X_engineered['worst perimeter']**2 / X_engineered['worst area']

# Symmetry ratios
X_engineered['mean symmetry ratio'] = X_engineered['mean symmetry'] / X_engineered['mean texture']
X_engineered['worst symmetry ratio'] = X_engineered['worst symmetry'] / X_engineered['worst texture']

# Fractal dimension differences
X_engineered['fractal dim diff'] = X_engineered['worst fractal dimension'] - X_engineered['mean fractal dimension']

# Concavity to area ratio
X_engineered['mean concavity area ratio'] = X_engineered['mean concavity'] / X_engineered['mean area']
X_engineered['worst concavity area ratio'] = X_engineered['worst concavity'] / X_engineered['worst area']

print("Original features:", X.shape[1])
print("Engineered features:", X_engineered.shape[1])

# Scale engineered data
X_engineered_scaled = scaler.fit_transform(X_engineered)

# LASSO on engineered features
lasso.fit(X_engineered_scaled, y)
selector = SelectFromModel(lasso, prefit=True)
X_selected = selector.transform(X_engineered_scaled)
selected_features = X_engineered.columns[selector.get_support()]

print("Selected engineered features:", selected_features.tolist())

## 3. Feature Selection & Engineering

In [ ]:
# Histograms of features
X.hist(bins=20, figsize=(20,15))
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12,10))
corr = pd.concat([X, y], axis=1).corr()
sns.heatmap(corr, annot=False, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Class distribution
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=pd.concat([X, y], axis=1))
plt.title('Class Distribution')
plt.show()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

print("Dataset shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

## 1. Data Loading & Preprocessing

# Cancer Prediction Model - Exploratory Data Analysis

This notebook performs exploratory data analysis on the Breast Cancer Wisconsin dataset.